In [2]:
!which python

/home/ec2-user/anaconda3/envs/python3/bin/python


In [3]:
!pip install cmake==3.27.0 pyarrow==16.1.0 datasets transformers accelerate sagemaker boto3 --upgrade --no-cache-dir

INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 473.5 MB/s  0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.10.0
    Uninstalling fsspec-2025.10.0:
      Successfully uninstalled fsspec-2025.10.0
  Attempting uninstall: botocore━━━━━━━━━━━━━━━━ 0/2 [fsspec]
    Found existing installation: botocore 1.40.700/2 [fsspec]
    Uninstalling botocore-1.40.70:90m╺━━━━━━━━━━━━━━━━━━━ 1/2 [botocore]
      Successfully uninstalled botocore-1.40.70━━━━━━━━━━━━━━━━━━━ 1/2 [botocore]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [botocore]1/2 [botocore]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.25.2 requires botocore<1.40.71,>=1.40.46, but you have botocore 1.40.74 w

In [4]:
!pip install peft

In [6]:
!pip install -U bitsandbytes

In [7]:
import boto3
from datasets import load_dataset

In [8]:
s3=boto3.client("s3")

In [9]:
response = s3.list_objects_v2(Bucket="llm-finetune-dataset-santosh", Prefix="datasets/")

In [10]:
for obj in response.get("Contents", []):
    print(obj)

{'Key': 'datasets/', 'LastModified': datetime.datetime(2025, 11, 15, 9, 52, 20, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 0, 'StorageClass': 'STANDARD'}
{'Key': 'datasets/pharma_instruction_data.csv', 'LastModified': datetime.datetime(2025, 11, 15, 9, 53, 3, tzinfo=tzlocal()), 'ETag': '"272a3e05b127a3adba02834123a080df"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 2475, 'StorageClass': 'STANDARD'}


In [11]:
for obj in response.get("Contents", []):
    print(obj["Key"])

datasets/
datasets/pharma_instruction_data.csv


In [12]:
dataset_path = "s3://llm-finetune-dataset-santosh/datasets/pharma_instruction_data.csv/"

In [13]:
from datasets import load_dataset

In [14]:
dataset = load_dataset("csv",data_files={"train":dataset_path},split="train")
# dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv",split="train")

In [15]:
dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

In [16]:
print(dataset)

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})


In [17]:
print(dataset[0])

{'instruction': 'Explain the mechanism of action of Metformin.', 'input': None, 'output': 'Metformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.'}


In [18]:
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

In [19]:
dataset = dataset.map(format_example)

In [20]:
dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 5
})

In [21]:
dataset['text'][0]

'### Instruction:\nExplain the mechanism of action of Metformin.\n### Input:\nNone\n### Response:\nMetformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.'

In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

In [26]:
model_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [27]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [28]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [29]:
def tokenize_fn(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [30]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [31]:
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
)

In [ ]:
trainer.train()

In [ ]:

model_path = "/content/tinyllama-instruction/checkpoint-3"

In [ ]:

instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [ ]:

prompt = "Explain the mechanism of action of Metformin."

In [ ]:

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:

outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))